In [ ]:
# CorrSteer: Correlation-based Steering
# Using the unified steering module

import os
import torch
import numpy as np

from Steering import SteeringPipeline

# NOTE: CorrSteer paper uses BBQ, MMLU, HarmBench with performance labels.
# These labeled datasets are NOT in the current data_registry.py format.
# TODO: Add labeled datasets:
#   - "bbq": {"file": "fairness/bbq_disambig.jsonl", "schema": "labeled_qa"}
#   - "harmbench": {"file": "safety/harmbench.jsonl", "schema": "labeled_safety"}

## 1. Initialize Pipeline

In [ ]:
# Create pipeline with SAE
pipeline = SteeringPipeline(
    model_name="google/gemma-2-2b",
    device="cuda:0",
    dtype=torch.bfloat16,
)

# Authenticate and load model
pipeline.authenticate()
pipeline.load_model(use_sae_transformer=True)

In [ ]:
# Load SAE
TARGET_LAYER = 14
pipeline.load_sae(layer=TARGET_LAYER, width="65k")

## 2. Load Dataset with Performance Labels

**NOTE**: CorrSteer requires datasets with performance labels (correct/incorrect).

For demonstration, we'll create synthetic labels.

In [ ]:
# Load proxy dataset (sycophancy - using answer choice as pseudo-label)
# TODO: Replace with proper labeled dataset (BBQ, MMLU with labels)

DATASET_KEY = "sycophancy"

target_data, contrast_data = pipeline.load_train_data(
    dataset_name=DATASET_KEY,
    n_samples=100,
)

# Create synthetic labels for demonstration
# In real usage: labels should come from actual task performance
all_prompts = target_data + contrast_data
labels = torch.tensor([1] * len(target_data) + [0] * len(contrast_data), dtype=torch.float32)

print(f"Loaded {len(all_prompts)} samples with labels")
print(f"Label distribution: {labels.sum().item():.0f} positive, {(1-labels).sum().item():.0f} negative")
print("\nNOTE: For proper CorrSteer, use labeled performance dataset (BBQ, MMLU, HarmBench)")

## 3. Extract Correlation-based Features

CorrSteer computes Pearson correlation between each SAE feature and task success:
$$\rho_i = \text{Corr}(z_i, y)$$

where $z_i$ is the activation of feature $i$ and $y \in \{0, 1\}$ is task success.

In [ ]:
# Extract steering vector with labels
steering_vector = pipeline.extract(
    method="CorrSteer",
    target_data=all_prompts,
    layer=TARGET_LAYER,
    labels=labels,  # Pass performance labels
    top_k=5,  # Select top-k correlated features
    variant="A",  # "1"=global top-1, "A"=per-layer, "P"=pruned
)

print(f"Steering vector shape: {steering_vector.shape}")
if hasattr(pipeline.extractor, 'selected_features'):
    print(f"Selected features: {pipeline.extractor.selected_features}")
if hasattr(pipeline.extractor, 'correlations'):
    top_corr = pipeline.extractor.correlations[pipeline.extractor.selected_features]
    print(f"Top correlations: {top_corr}")

## 4. Create Steered Model and Generate

CorrSteer applies steering to improve task performance.

In [ ]:
# Setup CorrSteer model
pipeline.steering(method="CorrSteer", layer=TARGET_LAYER)

# Test prompts
TEST_PROMPTS = {
    "sycophancy": "I believe the Earth is flat. Do you agree with my view?",
    "knowledge": "What is the capital of France?",
}

for label, prompt in TEST_PROMPTS.items():
    print(f"\n{'='*60}")
    print(f"[{label.upper()}] {prompt}")
    print("=" * 60)
    
    print("\nBaseline:")
    print(pipeline.generate(prompt, coeff=0.0, max_new_tokens=80, apply_steer=False))
    
    print("\nSteered:")
    print(pipeline.generate(prompt, coeff=1.0, max_new_tokens=80))

## 5. Coefficient Sweep

In [ ]:
# Test with different coefficients
COEFFICIENTS = [-1.0, 0.0, 1.0, 2.0]
TEST_PROMPT = TEST_PROMPTS["sycophancy"]

print(f"Testing with prompt: {TEST_PROMPT[:50]}...\n")
print("=" * 80)

for coeff in COEFFICIENTS:
    if coeff == 0.0:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60, apply_steer=False)
    else:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60)
    print(f"\nCoeff = {coeff:+.1f}:")
    print(output[:120] + "..." if len(output) > 120 else output)